# Asynchronous simulation for BMA

A short notebook exploring the dynamics of asynchronous networks

## load the libraries

In [1]:
import pybma

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np

from copy import deepcopy

## Define some functions for simulation and analysis

In [2]:
class StateEncoder:
    def __init__(self, model):
        maxes     = [v['RangeTo'] for v in model['Model']['Variables']]
        variables = [v['Id']      for v in model['Model']['Variables']]

        sorted_pairs    = sorted(zip(variables, maxes), key=lambda x: x[0])
        self.variables  = [v for v, _ in sorted_pairs]
        self.max_levels = [m for _, m in sorted_pairs]

        # Pure Python ints — no numpy, no overflow
        self.strides = [1]
        for m in self.max_levels[:-1]:
            self.strides.append(self.strides[-1] * (m + 1))

    def encode(self, state: tuple) -> int:
        return sum(v * s for v, s in zip(state, self.strides))

    def decode(self, code: int) -> tuple:
        result = []
        for m in self.max_levels:
            result.append(code % (m + 1))
            code //= (m + 1)
        return tuple(result)

In [3]:
class BNContext:
    """
    Holds the canonical variable ordering derived from the model.
    All state conversions go through this object, eliminating
    ad-hoc variable list construction elsewhere.
    """
    def __init__(self, model):
        self.encoder  = StateEncoder(model)
        self.variables  = self.encoder.variables    # same sorted list, single source
        self.max_levels = self.encoder.max_levels   # same sorted list, single source

    def stateToTuple(self, state: dict) -> tuple:
        return tuple(state[v] for v in self.variables)

    def tupleToState(self, t: tuple) -> dict:
        return dict(zip(self.variables, t))

    def async_step(self, qn, state: dict) -> set[tuple]:
        sync_result = pybma.simulate(qn, steps=3, initial_values=state)
        sync_target = {v: sync_result[v][-1] for v in self.variables}
        
        async_result = set()
        for v in self.variables:
            if sync_target[v] != state[v]:
                successor = dict(state)
                successor[v] = sync_target[v]
                async_result.add(self.stateToTuple(successor))
        return async_result

In [4]:
from collections import deque

def exists_order_before(
    graph: dict[int, set[int]],
    ctx: BNContext,
    start: tuple,
    i: int, v: int,
    j: int, w: int
) -> bool:
    """
    Returns True if there exists a path from start where
    gene i takes value v at some state BEFORE gene j takes value w.
    """
    start_code = ctx.encoder.encode(start)
    
    visited = set()
    queue = deque()
    queue.append((start_code, ctx.encoder.decode(start_code)[i] == v))
    
    while queue:
        code, i_seen = queue.popleft()
        
        if (code, i_seen) in visited:
            continue
        visited.add((code, i_seen))
        
        state = ctx.encoder.decode(code)
        
        if state[j] == w and not i_seen:
            continue
        
        if i_seen and state[j] != w:
            return True
        
        for successor_code in graph.get(code, set()):
            new_i_seen = i_seen or (ctx.encoder.decode(successor_code)[i] == v)
            queue.append((successor_code, new_i_seen))
    
    return False

In [5]:
def exists_order_before_online(
    qn: BNModel,
    ctx: BNContext,
    start: tuple,
    i: int, v: int,
    j: int, w: int
) -> tuple[bool, list[tuple] | None]:
    """
    Returns (True, trace) if there exists a path from start where
    gene i takes value v before gene j takes value w, or (False, None).
    trace is a list of decoded state tuples from start to the witnessing state.
    """
    start_code = ctx.encoder.encode(start)
    initial_i_seen = start[i] == v
        
    visited = set()
    # parent maps (code, i_seen) -> (parent_code, parent_i_seen) | None
    parent: dict[tuple[int, bool], tuple[int, bool] | None] = {}

    queue = deque()
    queue.append((start_code, initial_i_seen))
    parent[(start_code, initial_i_seen)] = None

    while queue:
        code, i_seen = queue.popleft()

        if (code, i_seen) in visited:
            continue
        visited.add((code, i_seen))

        state = ctx.encoder.decode(code)

        if state[j] == w and not i_seen:
            continue

        if i_seen and state[j] != w:
            return True, _reconstruct_trace(parent, ctx, code, i_seen)

        #print("P:",state[i],state[j])
        for successor in ctx.async_step(qn, ctx.tupleToState(state)):
            #print("S:",successor[i],successor[j])
            successor_code = ctx.encoder.encode(successor)
            new_i_seen = i_seen or (successor[i] == v)
            key = (successor_code, new_i_seen)
            if key not in parent:
                parent[key] = (code, i_seen)
            queue.append((successor_code, new_i_seen))

    return False, None


def _reconstruct_trace(
    parent: dict[tuple[int, bool], tuple[int, bool] | None],
    ctx: BNContext,
    code: int,
    i_seen: bool
) -> list[tuple]:
    """Walk parent pointers back to the start and reverse."""
    trace = []
    cursor: tuple[int, bool] | None = (code, i_seen)
    while cursor is not None:
        trace.append(ctx.encoder.decode(cursor[0]))
        cursor = parent[cursor]
    trace.reverse()
    return trace

In [6]:
def async_simulate(qn,initial_values=None, ctx=None):
    if initial_values == None or ctx == None: raise
    
    variables = list(initial_values.keys())
    initial_state = ctx.stateToTuple(initial_values)
    
    graph: dict[int, set[int]] = {}

    # Adding a transition
    def add_transition(graph, from_state, to_state):
        graph.setdefault(ctx.encoder.encode(from_state), set()).add(ctx.encoder.encode(to_state))
    
    final_values = [initial_state]
    stop = False
    while not stop:
        stop = True
        updated_final_values = []
        for final_values_state in final_values:
            if ctx.encoder.encode(final_values_state) in graph.keys():
                continue
            stop = False
            final_values_single = ctx.tupleToState(final_values_state)
            step_result = ctx.async_step(qn,final_values_single)
            updated_final_values.append(step_result)
            for successor in step_result:
                add_transition(graph,final_values_state,successor)
        final_values = [x for sublist in updated_final_values for x in sublist]  
    return(graph)

In [7]:
def report_transitions(trace: list[tuple]) -> None:
    """
    Given a list of state tuples (e.g. from exists_order_before_online),
    prints the indices that change and by how much at each step.
    """
    for step, (src, tgt) in enumerate(zip(trace, trace[1:])):
        diffs = [(i, src[i], tgt[i], tgt[i] - src[i])
                 for i in range(len(src))
                 if src[i] != tgt[i]]
        print(f"Step {step}: {src} -> {tgt}")
        for i, old, new, delta in diffs:
            print(f"  index {i}: {old} -> {new}  (delta {delta:+d})")
        if len(diffs) != 1:
            print(f"  *** WARNING: {len(diffs)} genes changed (expected 1 for async) ***")

## load the model and check the stability

In [8]:
m = pybma.load_model("./ToyModelStable.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)
print(p)

{'ProofProgression': {'stable': True, 'history': [(4, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (3, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (2, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (1, {1: (0, 0), 2: (0, 0), 3: (0, 4)}), (0, {1: (0, 0), 2: (0, 4), 3: (0, 4)})]}, 'CounterExample': None}


In [9]:
final_state = dict( [ (varid,p['ProofProgression']['history'][0][1][varid][0]) for varid in p['ProofProgression']['history'][0][1].keys() ] )

In [10]:
# Some issue with number of steps; 3 gives us two states
pybma.simulate(qn,steps=3,initial_values=final_state)

{1: [0, 0], 2: [0, 0], 3: [0, 0]}

## Mutate the model

In [11]:
m['Model']['Variables'][0]['Formula'] = "2"
qM = pybma.model_to_qn(m)
pM = pybma.check_stability(qM)

print("###Modified Model###")
print(pM)

###Modified Model###
{'ProofProgression': {'stable': True, 'history': [(4, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (3, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (2, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (1, {1: (2, 2), 2: (2, 2), 3: (0, 4)}), (0, {1: (2, 2), 2: (0, 4), 3: (0, 4)})]}, 'CounterExample': None}


### Simulate the new model from the old stable state

In [12]:
pybma.simulate(qM,steps=3,initial_values=final_state)

{1: [0, 1], 2: [0, 0], 3: [0, 0]}

In [13]:
ctx = BNContext(m)

In [14]:
ctx.async_step(qM,final_state)

{(1, 0, 0)}

In [15]:
graph = async_simulate(qM,final_state,ctx)
print(graph)

{0: {1}, 1: {2, 6}, 6: {31, 7}, 2: {7}, 7: {32, 12}, 31: {32}, 32: {37}, 12: {37}, 37: {62}}


In [16]:
variables = list(final_state.keys())



In [17]:
exists_order_before(graph, ctx, (0, 0, 0), 0, 1, 1, 1)

True

In [18]:
exists_order_before(graph, ctx, (0, 0, 0), 1, 1, 0, 1)

False

In [19]:
variables = list(final_state.keys())
exists_order_before_online(qM, ctx, (0, 0, 0), 0, 2, 1, 1)

(True, [(0, 0, 0), (1, 0, 0), (2, 0, 0)])

In [20]:
exists_order_before_online(qM, ctx, (0, 0, 0), 1, 1, 0, 1)

(False, None)

In [21]:
finding, trace = exists_order_before_online(qM, ctx, (0, 0, 0), 0, 2, 1, 1)
report_transitions(trace)

Step 0: (0, 0, 0) -> (1, 0, 0)
  index 0: 0 -> 1  (delta +1)
Step 1: (1, 0, 0) -> (2, 0, 0)
  index 0: 1 -> 2  (delta +1)


## B cell differentiation

In [22]:
m = pybma.load_model("./B-cell-undifferentiated-stable.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)
print("Stable =", p['ProofProgression']['stable'])

Stable = True


In [23]:
final_state = dict( [ (varid,p['ProofProgression']['history'][0][1][varid][0]) for varid in p['ProofProgression']['history'][0][1].keys() ] )

In [24]:
variable_names_lookup = { v['Id']:v['Name'] for v in m['Model']['Variables'] }
variable_ids_lookup = { v['Name']:v['Id'] for v in m['Model']['Variables'] }
print('CLP',variable_ids_lookup['CLP'])
print('Naive B cell',variable_ids_lookup['Naive B cell'])

CLP 146
Naive B cell 164


In [25]:
print(final_state)

{2: 0, 3: 0, 7: 0, 8: 0, 11: 0, 17: 0, 19: 0, 30: 0, 38: 0, 59: 2, 68: 0, 74: 0, 120: 1, 121: 1, 122: 1, 126: 0, 130: 0, 133: 0, 146: 1, 147: 0, 148: 0, 149: 0, 164: 0, 171: 0, 179: 0, 180: 0, 181: 0, 182: 0, 198: 0, 206: 0, 211: 0, 231: 2, 232: 0, 236: 0, 237: 0, 247: 0, 254: 0, 257: 0, 266: 0, 272: 1, 275: 0, 280: 1, 293: 0, 307: 0, 313: 0, 322: 0, 331: 0, 343: 0, 344: 0, 366: 0, 383: 0, 390: 0, 392: 0, 408: 0, 414: 0, 415: 0, 423: 0, 424: 0, 425: 0, 437: 0, 457: 0, 464: 0, 471: 0, 472: 0, 473: 0}


In [26]:
print("CLP final =", final_state[variable_ids_lookup['CLP']])
print("Naive B cell final =", final_state[variable_ids_lookup['Naive B cell']])

CLP final = 1
Naive B cell final = 0


In [27]:
m['Model']['Variables'][0]

{'Name': 'Ilr7',
 'Id': 2,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'min(2,(2*var(7)))+var(74)+var(8)-(4*var(30))-max(0,(2*var(206)-var(211)))'}

In [28]:

varlist = [v['Name'] for v in m['Model']['Variables'] ]
variableDict = dict(zip(varlist,range(len(varlist))))

In [29]:
variableDict

{'Ilr7': 0,
 'Il7': 1,
 'E2a': 2,
 'Foxo1': 3,
 'Pax5': 4,
 'Ikzf1': 5,
 'CD19': 6,
 'BCR': 7,
 'RAG': 8,
 'RUNX1': 9,
 'Vpreb1': 10,
 'EBF1': 11,
 'Proliferation': 12,
 'Apoptosis': 13,
 'Cycle-arrest': 14,
 'CCND3': 15,
 'Bcl2': 16,
 'TSLP': 17,
 'CLP': 18,
 'PreproB': 19,
 'ProB': 20,
 'PreB': 21,
 'Naive B cell': 22,
 'JAK-STAT': 23,
 'Syk': 24,
 'Bcl6': 25,
 'CDKN1A': 26,
 'CDKN1B': 27,
 'Bok': 28,
 'preBCR': 29,
 'Bach2': 30,
 'CXCL12': 31,
 'CXCR4': 32,
 'Id3': 33,
 'Ikzf3': 34,
 'Igll1': 35,
 'CRLF2': 36,
 'Myc': 37,
 'PIP3': 38,
 'CCND2': 39,
 'BCR-ABL': 40,
 'CDKN2A': 41,
 'preBCR_Pax5': 42,
 'AKT': 43,
 'DNA-damage': 44,
 'Bim': 45,
 'p53': 46,
 'MAPK': 47,
 'ERK1_2': 48,
 'Blnk': 49,
 'CXCR4_CXCL12': 50,
 'KO_preBCR': 51,
 'KO_JAK_STAT': 52,
 'IKZF1_ko': 53,
 'BCL6_KO': 54,
 'MYC_KO': 55,
 'KO_CDKN1A': 56,
 'KO_CDKN1B': 57,
 'KO_CDKN2A': 58,
 'Force-Pax5': 59,
 'EBF1_KO': 60,
 'PAX5_KO': 61,
 'KO_PIP3': 62,
 'KO_MAPK': 63,
 'KO_BCRABL': 64}

In [30]:
m['Model']['Variables'][variableDict['PreproB']]

{'Name': 'PreproB',
 'Id': 147,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'min(1,var(2))-max(0,(2*var(74)-2))-max(0,2*var(206)-2)-var(30)'}

In [31]:
m['Model']['Variables'][variableDict['CLP']]

{'Name': 'CLP',
 'Id': 146,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': '1-var(2)-var(30)-var(206)'}

In [32]:
m['Model']['Variables'][variableDict['Il7']]['RangeTo'] = 2
m['Model']['Variables'][variableDict['Ikzf1']]['RangeTo'] = 2
qM = pybma.model_to_qn(m)
#pM = pybma.check_stability(qM)

#print("###Modified Model###")
#print(pM)

In [33]:
ctx = BNContext(m)
final_state_tuple = ctx.stateToTuple(final_state)
clpVar = ctx.variables.index(146)
preprobVar = ctx.variables.index(147)

In [34]:
# Quick sanity check at the call site
assert set(final_state.keys()) == set(ctx.variables), (
    f"Variable mismatch:\n"
    f"  stability: {sorted(final_state.keys())}\n"
    f"  ctx:       {sorted(ctx.variables)}"
)
print("Start state:")
for v, val in zip(ctx.variables, final_state_tuple):
    print(f"  {v}: {val}")

Start state:
  2: 0
  3: 0
  7: 0
  8: 0
  11: 0
  17: 0
  19: 0
  30: 0
  38: 0
  59: 2
  68: 0
  74: 0
  120: 1
  121: 1
  122: 1
  126: 0
  130: 0
  133: 0
  146: 1
  147: 0
  148: 0
  149: 0
  164: 0
  171: 0
  179: 0
  180: 0
  181: 0
  182: 0
  198: 0
  206: 0
  211: 0
  231: 2
  232: 0
  236: 0
  237: 0
  247: 0
  254: 0
  257: 0
  266: 0
  272: 1
  275: 0
  280: 1
  293: 0
  307: 0
  313: 0
  322: 0
  331: 0
  343: 0
  344: 0
  366: 0
  383: 0
  390: 0
  392: 0
  408: 0
  414: 0
  415: 0
  423: 0
  424: 0
  425: 0
  437: 0
  457: 0
  464: 0
  471: 0
  472: 0
  473: 0


In [35]:
finding, trace = exists_order_before_online(qM, ctx, final_state_tuple, preprobVar, 1, clpVar, 1)
print(finding)
for state in trace:
    s = ctx.tupleToState(state)
    print(s[147],s[146])

False


TypeError: 'NoneType' object is not iterable

In [36]:
trace

In [37]:
report_transitions(trace)

TypeError: 'NoneType' object is not subscriptable

In [38]:
step = ctx.async_step(qM,final_state)

In [39]:
len(step)

2

In [40]:
successors = [s for s in step]

In [41]:
list(zip(ctx.stateToTuple(final_state),successors[0]))

[(0, 0),
 (0, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (2, 2),
 (0, 0),
 (0, 0),
 (1, 1),
 (1, 1),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (2, 2),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (0, 0),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0)]

In [42]:
list(zip(ctx.stateToTuple(final_state),successors[1]))

[(0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (2, 2),
 (0, 0),
 (0, 0),
 (1, 1),
 (1, 1),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (2, 2),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (0, 0),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0)]

In [43]:
trace

In [44]:
ctx.encoder.encode(ctx.stateToTuple(final_state))

25017678949485485412

In [45]:
ctx.encoder.encode(successors[0])

25017678949485485415

In [46]:
def encode_decode(ctx, t):
    code = ctx.encoder.encode(ctx.stateToTuple(t))
    return(ctx.encoder.decode(code))

list(zip(ctx.stateToTuple(final_state),encode_decode(ctx, final_state)))

[(0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (2, 2),
 (0, 0),
 (0, 0),
 (1, 1),
 (1, 1),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (2, 2),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (0, 0),
 (1, 1),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0)]

In [47]:
def diagnose_encoder(ctx):
    print("=== Variable ordering ===")
    for i, (v, m) in enumerate(zip(ctx.variables, ctx.max_levels)):
        print(f"  [{i}] {v}  max={m}  stride={ctx.encoder.strides[i]}")

    print("\n=== Encode/decode round-trip ===")
    # Test with a simple state: all zeros
    zero = tuple(0 for _ in ctx.max_levels)
    code = ctx.encoder.encode(zero)
    back = ctx.encoder.decode(code)
    print(f"  zero:    {zero}")
    print(f"  encoded: {code}")
    print(f"  decoded: {back}")
    print(f"  match:   {zero == back}")

    # Test with max state
    maxs = tuple(ctx.max_levels)
    code = ctx.encoder.encode(maxs)
    back = ctx.encoder.decode(code)
    print(f"  max:     {maxs}")
    print(f"  encoded: {code}")
    print(f"  decoded: {back}")
    print(f"  match:   {maxs == back}")

    # Test with final_state
    print("\n=== Final state round-trip ===")
    t = ctx.stateToTuple(final_state)
    code = ctx.encoder.encode(t)
    back = ctx.encoder.decode(code)
    print(f"  input tuple: {t}")
    print(f"  decoded:     {back}")
    for i, (v, a, b) in enumerate(zip(ctx.variables, t, back)):
        match = "OK" if a == b else "*** MISMATCH ***"
        print(f"  [{i}] {v}: {a} -> {b}  {match}")

diagnose_encoder(ctx)

=== Variable ordering ===
  [0] 2  max=2  stride=1
  [1] 3  max=2  stride=3
  [2] 7  max=2  stride=9
  [3] 8  max=2  stride=27
  [4] 11  max=2  stride=81
  [5] 17  max=2  stride=243
  [6] 19  max=2  stride=729
  [7] 30  max=2  stride=2187
  [8] 38  max=2  stride=6561
  [9] 59  max=2  stride=19683
  [10] 68  max=2  stride=59049
  [11] 74  max=2  stride=177147
  [12] 120  max=4  stride=531441
  [13] 121  max=4  stride=2657205
  [14] 122  max=4  stride=13286025
  [15] 126  max=2  stride=66430125
  [16] 130  max=2  stride=199290375
  [17] 133  max=0  stride=597871125
  [18] 146  max=2  stride=597871125
  [19] 147  max=2  stride=1793613375
  [20] 148  max=2  stride=5380840125
  [21] 149  max=2  stride=16142520375
  [22] 164  max=2  stride=48427561125
  [23] 171  max=2  stride=145282683375
  [24] 179  max=2  stride=435848050125
  [25] 180  max=2  stride=1307544150375
  [26] 181  max=2  stride=3922632451125
  [27] 182  max=2  stride=11767897353375
  [28] 198  max=2  stride=35303692060125
  [2

In [ ]:
graph = async_simulate(qM,final_state,ctx)
